# AION Chat — Conversational Fine-Tune on Colab

**Runtime required:** T4 GPU — `Runtime → Change runtime type → T4 GPU`

**Before running:** add Colab secrets (sidebar → 🔑 Key icon → Add new secret):
- `GITHUB_TOKEN` — fine-grained GitHub PAT with read access to your repo
- `GITHUB_USERNAME` — your GitHub username

Run cells top-to-bottom. Each cell is idempotent — safe to re-run after a session reset.

In [ ]:
# Cell 1 — Install dependencies & clone repo
import subprocess, sys, os
from google.colab import userdata

TOKEN           = userdata.get('GITHUB_TOKEN')
GITHUB_USERNAME = userdata.get('GITHUB_USERNAME')

if not GITHUB_USERNAME:
    raise ValueError("Add a Colab secret named 'GITHUB_USERNAME' (sidebar → 🔑 Key icon).")
if not TOKEN:
    raise ValueError("Add a Colab secret named 'GITHUB_TOKEN' (sidebar → 🔑 Key icon).")

REPO_URL = f'https://{TOKEN}@github.com/{GITHUB_USERNAME}/aion.git'

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'mamba-ssm', 'causal-conv1d',
    'datasets', 'tokenizers', 'pyyaml', 'tqdm', 'bitsandbytes',
], check=False)

if not os.path.exists('/content/aion'):
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, '/content/aion'], check=True)
    print('Repo cloned.')
else:
    subprocess.run(['git', '-C', '/content/aion', 'pull', '--ff-only'], check=True)
    print('Repo up to date.')

if '/content/aion/src' not in sys.path:
    sys.path.insert(0, '/content/aion/src')
print('Done.')

In [ ]:
# Cell 2 — Mount Google Drive (checkpoints will be saved here)
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

CKPT_DIR = Path('/content/drive/MyDrive/aion_checkpoints/mamba_chat')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Checkpoint dir: {CKPT_DIR}')

In [ ]:
# Cell 3 — Download chat datasets (~25 MB total)
# oasst1:    ~9k best-path multi-turn conversations  (~20 MB)
# dolly-15k: 15k single-turn Q&A                    (~13 MB, adds instruction diversity)
import sys, runpy
from pathlib import Path

# Flush stale llm_lab module cache so git-pulled changes take effect
for _key in list(sys.modules.keys()):
    if _key.startswith('llm_lab'):
        del sys.modules[_key]

RAW_DIR = '/content/data/raw'

sys.argv = ['cli', 'download', '--target', RAW_DIR, '--preset', 'chat']
runpy.run_module('llm_lab.cli', run_name='__main__', alter_sys=True)
print('Download complete.')


In [ ]:
# Cell 4 — Merge datasets into a single chat JSON + train tokenizer
import sys, runpy, json
from pathlib import Path

RAW_DIR = Path('/content/data/raw')
DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

SEED_PATH   = '/content/aion/src/llm_lab/data/instruction_seed.json'
MERGED_PATH = '/content/data/chat_merged.json'

# --- Merge all sources via CLI (oasst1 + dolly-15k + AION identity examples) ---
sys.argv = [
    'cli', 'merge-chat',
    '--raw-dir', str(RAW_DIR),
    '--out',     MERGED_PATH,
    '--seed',    SEED_PATH,
]
runpy.run_module('llm_lab.cli', run_name='__main__', alter_sys=True)

# --- Build corpus for tokenizer training (all message text concatenated) ---
corpus_path = DATA_DIR / 'corpus.txt'
chat_examples = json.loads(Path(MERGED_PATH).read_text())
with open(corpus_path, 'w', encoding='utf-8') as f:
    for ex in chat_examples:
        for msg in ex['messages']:
            f.write(msg['content'].strip() + '\n')
print(f'Corpus for tokenizer: {corpus_path.stat().st_size / 1024 / 1024:.1f} MB')

# --- Train BPE tokenizer (vocab_size=16384) ---
TOKENIZER_PATH = '/content/data/tokenizer.json'
sys.argv = ['cli', 'tokenizer', '--corpus', str(corpus_path), '--out', TOKENIZER_PATH, '--vocab-size', '16384']
runpy.run_module('llm_lab.cli', run_name='__main__', alter_sys=True)
print(f'Tokenizer saved: {TOKENIZER_PATH}')


In [ ]:
# Cell 5 — Keep-alive + Drive logger
# - Tees all print() output to CKPT_DIR/training.log on Drive
# - Writes a human-readable CKPT_DIR/status.txt every 60s from metrics.json
import sys, time, threading, json
from datetime import datetime
from pathlib import Path
from tqdm import tqdm as _tqdm

LOG_PATH    = CKPT_DIR / 'training.log'
STATUS_PATH = CKPT_DIR / 'status.txt'

class _Tee:
    """Write to both the original stdout and a Drive log file."""
    def __init__(self, original, log_path):
        self._orig = original
        self._log  = open(log_path, 'a', encoding='utf-8', buffering=1)
    def write(self, data):
        self._orig.write(data)
        self._log.write(data)
    def flush(self):
        self._orig.flush()
        self._log.flush()
    def __getattr__(self, attr):
        return getattr(self._orig, attr)

sys.stdout = _Tee(sys.stdout, LOG_PATH)
print(f'\n=== Session started {datetime.now().strftime("%Y-%m-%d %H:%M:%S")} ===')

def _monitor():
    metrics_path = CKPT_DIR / 'metrics.json'
    while True:
        time.sleep(60)
        try:
            if metrics_path.exists():
                data = json.loads(metrics_path.read_text())
                train = [e for e in data if 'train_loss' in e]
                val   = [e for e in data if 'val_loss' in e]
                lines = [
                    f'Updated:    {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
                    f'Step:       {train[-1]["step"] if train else "n/a"}',
                    f'Train loss: {train[-1]["train_loss"]:.4f}' if train else 'Train loss: n/a',
                    f'Val loss:   {val[-1]["val_loss"]:.4f} (step {val[-1]["step"]})' if val else 'Val loss: n/a',
                    f'Best val:   {min(e["val_loss"] for e in val):.4f} @ step {min(val, key=lambda x: x["val_loss"])["step"]}' if val else 'Best val: n/a',
                    f'Elapsed:    {train[-1]["elapsed_s"] / 3600:.2f}h' if train and 'elapsed_s' in train[-1] else '',
                ]
                STATUS_PATH.write_text('\n'.join(lines) + '\n')
                _tqdm.write(f'[monitor] {lines[0]} | {lines[1]} | {lines[2]}')
        except Exception:
            pass

threading.Thread(target=_monitor, daemon=True).start()
print(f'Drive logger active — log: {LOG_PATH.name}, status: {STATUS_PATH.name}')


In [ ]:
# Cell 6 — Train (single session — uses the full ~11h Colab window)
# Targets ~15,000 steps which fits in one session with setup overhead.
# If interrupted: re-run cells 1-5, then re-run this cell — resumes from last Drive checkpoint.
import sys, runpy, yaml, shutil
from pathlib import Path

cfg = yaml.safe_load(
    Path('/content/aion/src/llm_lab/configs/mamba_chat.yaml').read_text()
)

cfg['dataset_type']     = 'chat'
cfg['instruction_data'] = '/content/data/chat_merged.json'
cfg['train_path']       = ''
cfg['val_path']         = ''
cfg['tokenizer_path']   = '/content/data/tokenizer.json'
cfg['checkpoint_dir']   = str(CKPT_DIR)
cfg['max_steps']        = 15000   # ~6h on T4; leaves buffer before the 12h session cap
cfg['checkpoint_every'] = 1000   # save to Drive every 1000 steps

run_cfg_path = '/content/run.yaml'
Path(run_cfg_path).write_text(yaml.dump(cfg))

shutil.copy2('/content/data/tokenizer.json', CKPT_DIR / 'tokenizer.json')
shutil.copy2(run_cfg_path, CKPT_DIR / 'run.yaml')

print(f'Training mamba_chat for {cfg["max_steps"]} steps (~6h)...')
print(f'Checkpoints -> {CKPT_DIR}')
print()

sys.argv = ['cli', 'train', '--config', run_cfg_path]
runpy.run_module('llm_lab.cli', run_name='__main__', alter_sys=True)


In [ ]:
# Cell 7 — Test the trained model with a sample conversation
import torch
from pathlib import Path
from tokenizers import Tokenizer

sys.path.insert(0, '/content/aion/src')
from llm_lab.training.config import TrainConfig
from llm_lab.training.model_factory import build_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = Tokenizer.from_file('/content/data/tokenizer.json')
cfg = TrainConfig.load(Path('/content/run.yaml'))

# Load latest checkpoint
ckpt_path = CKPT_DIR / 'latest.pt'
model = build_model(cfg).to(device)
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'], strict=False)
model.eval()
print(f'Loaded checkpoint from step {ckpt["step"]}')

def chat(user_message: str, max_new_tokens: int = 200, temperature: float = 0.8) -> str:
    prompt = f'<|user|>{user_message}<|end|>\n<|assistant|>'
    input_ids = torch.tensor([tokenizer.encode(prompt).ids], dtype=torch.long).to(device)
    end_id = tokenizer.encode('<|end|>').ids[0]

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(input_ids)
            next_logits = logits[:, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            input_ids = torch.cat([input_ids, next_id], dim=1)
            if next_id.item() == end_id:
                break

    generated = tokenizer.decode(input_ids[0].tolist())
    # Extract only the assistant response
    marker = '<|assistant|>'
    if marker in generated:
        generated = generated.split(marker, 1)[1].replace('<|end|>', '').strip()
    return generated

# Try it out
print('User: What is machine learning?')
print('Assistant:', chat('What is machine learning?'))